In [1]:
import gzip,re, roman 
import numpy as np
import pandas as pd
import gffpandas.gffpandas as gffpd

from tqdm import tqdm
from Bio import SeqIO

tqdm.pandas()

In [2]:
def format_chrom(x):
    if '_' not in x :
        x = x.replace('CHR', 'CHR_')
        return x
    return x

def format_roman_chrom(x):
    if 'MT' not in x : 
        rom_nb = x.split('_')[1]
        nb = roman.fromRoman(rom_nb)
        x = 'CHR_'+ str(nb)
        return x 
    return x

def format_numeric_chrom(x):
    if 'MT' not in x : 
        num_nb = x.replace('CHR','')
        nb = roman.toRoman(int(num_nb))
        x = 'chr'+ str(nb)
        return x 
    return 'chrXVII'

## Promoter Annotations

In [3]:
genome_annot_rossi_folder = '../../data/genome_annotations/rossi_et_al_2021'
PROMOTER_TABLE = pd.read_csv(f'{genome_annot_rossi_folder}/nfr_ndr_feature_positions.csv').rename(columns={"Chromosome_Name":"seq_id", "systematic_id":"locus_id", "common_name":"name", "NFR/NDR_Start":"start", "NFR/NDR_End":"end", "nucleosome_stability":"description"})
PROMOTER_TABLE['seq_id'] = PROMOTER_TABLE['seq_id'].apply(format_numeric_chrom)
PROMOTER_TABLE['source'] = 'rossi_2021_promoters'
PROMOTER_TABLE['type'] = 'promoter'
PROMOTER_TABLE['score'] = 1
PROMOTER_TABLE['phase'] = '.'
PROMOTER_TABLE['name'] = PROMOTER_TABLE['name'].replace('IMP2\'','IMP21')
PROMOTER_TABLE['attributes'] = [f"Name={PROMOTER_TABLE['name'][idx]};Desc={PROMOTER_TABLE['description'][idx]}" for idx in PROMOTER_TABLE.index ]
prom_plus = PROMOTER_TABLE[PROMOTER_TABLE['strand'] == '+'].reset_index(drop=True)
prom_minus = PROMOTER_TABLE[PROMOTER_TABLE['strand'] == '-'].rename(columns={"start":"end", "end":"start"}).reset_index(drop=True)

PROMOTER_TABLE = pd.concat([prom_plus, prom_minus]).reset_index(drop=True)
display(PROMOTER_TABLE)
# PROMOTER_TABLE[['seq_id', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes', 'locus_id']].to_csv(f'{genome_annot_rossi_folder}/Rossi_2021_Promoters_V64.csv', index=False)

,seq_id,strand,feature_type,gene_class,husinga_classification,locus_id,name,start,end,NFR/NDR_Length,description,TATA_type,source,type,score,phase,attributes
0,chrIV,+,01_RP,01_RP_NFR/NDR,RP,YDR382W_NFR/NDR,RPP2B,1238654.0,1239404.0,750.0,fragile,TATA_like,rossi_2021_promoters,promoter,1,.,Name=RPP2B;Desc=fragile
1,chrIV,+,01_RP,01_RP_NFR/NDR,RP,YDR418W_NFR/NDR,RPL12B,1301255.0,1301670.0,415.0,fragile,TATA_like,rossi_2021_promoters,promoter,1,.,Name=RPL12B;Desc=fragile
2,chrVIII,+,01_RP,01_RP_NFR/NDR,RP,YHL015W_NFR/NDR,RPS20,74997.0,75429.0,432.0,fragile,TATA_like,rossi_2021_promoters,promoter,1,.,Name=RPS20;Desc=fragile
3,chrXII,+,01_RP,01_RP_NFR/NDR,RP,YLR167W_NFR/NDR,RPS31,498112.0,498862.0,750.0,fragile,TATA_like,rossi_2021_promoters,promoter,1,.,Name=RPS31;Desc=fragile
4,chrX,+,01_RP,01_RP_NFR/NDR,RP,YJL189W_NFR/NDR,RPL39,75147.0,75897.0,750.0,fragile,TATA_like,rossi_2021_promoters,promoter,1,.,Name=RPL39;Desc=fragile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8071,chrVI,-,21_TEL,Telomere_NFR/NDR,NaN,TEL06L_NFR/NDR,NaN,5530.0,5611.0,81.0,NaN,NaN,rossi_2021_promoters,promoter,1,.,Name=nan;Desc=nan
8072,chrV,-,21_TEL,Telomere_NFR/NDR,NaN,TEL05L_NFR/NDR,NaN,6473.0,6554.0,81.0,NaN,NaN,rossi_2021_promoters,promoter,1,.,Name=nan;Desc=nan
8073,chrII,-,21_TEL,Telomere_NFR/NDR,NaN,TEL02L_NFR/NDR,NaN,6680.0,6850.0,170.0,NaN,NaN,rossi_2021_promoters,promoter,1,.,Name=nan;Desc=nan
8074,chrXII,-,21_TEL,Telomere_NFR/NDR,NaN,TEL12L_NFR/NDR,NaN,12085.0,12312.0,227.0,NaN,NaN,rossi_2021_promoters,promoter,1,.,Name=nan;Desc=nan


### SGD information table

In [4]:
genome_annot_sgd_folder = '../../data/genome_annotations/sgd_database/original'

### OTHER FEATURES
others_annotations = []
fasta_gz = f'{genome_annot_sgd_folder}/other_features_genomic_R64-3-1_20210421.fasta.gz'
with gzip.open(fasta_gz, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ID, location, genome_version, others_type = record.description.split(', ')[:4]
        others_id = ID.split(' ')[0]
        others_name = ID.split(' ')[1]
        sgd_id = ID.split(' ')[2]
        res = re.findall(r'(Chr [A-Za-z]+) from ([0-9]+)-([0-9]+)', str(location))[0]
        Chr = res[0].upper().replace(' ','_').replace('MITO', 'MT')
        description = ' ; '.join(record.description.split(', ')[4:])
        if 'reverse complement' == others_type:
            Start = res[2]
            End = res[1]
            cds_type = description.split(' ; ')[0]
            description = ' ; '.join(description.split(' ; ')[1:])
        else : 
            Start = res[1]
            End = res[2]
        
        others_annotations.append([others_id, others_name, sgd_id, others_type, Chr, Start, End, description, genome_version])

OTHERS_TABLE = pd.DataFrame(others_annotations, columns=['locus_id', 'name', 'sgd_id', 'genome_annotations', 'chrom', 'start', 'end', 'description', 'genome_version'])
OTHERS_TABLE['chrom'] = OTHERS_TABLE['chrom'].apply(format_roman_chrom)
display(OTHERS_TABLE)
# OTHERS_TABLE.to_csv('../../data/genome_annotations/sgd_database/other_features_genomic_R64-3-1.csv', index=False)
OTHERS_dict = OTHERS_TABLE.set_index('locus_id').to_dict()


### RNA
rna_annotations = []
fasta_gz = f'{genome_annot_sgd_folder}/rna_coding_R64-3-1_20210421.fasta.gz'
with gzip.open(fasta_gz, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ID, location, genome_version, rna_type = record.description.split(', ')[:4]
        rna_id = ID.split(' ')[0]
        rna_name = ID.split(' ')[1]
        sgd_id = ID.split(' ')[2]
        res = re.findall(r'(Chr [A-Za-z]+) from ([0-9]+)-([0-9]+)', str(location))[0]
        Chr = res[0].upper().replace(' ','_').replace('MITO', 'MT')
        description = (' ; '.join(record.description.split(', ')[4:]))
        if 'reverse complement' == rna_type:
            Start = res[2]
            End = res[1]
            try : 
                rna_type = re.findall(r'([A-Za-z0-9]+\-)\)',str(description.split(' ; ')[0]))[0]
            except : 
                rna_type = str(description.split(' ; ')[0])
        else : 
            Start = res[1]
            End = res[2]
        
        rna_annotations.append([rna_id, rna_name, sgd_id, rna_type, Chr, Start, End, description, genome_version])

RNA_TABLE = pd.DataFrame(rna_annotations, columns=['locus_id', 'name', 'sgd_id', 'genome_annotations', 'chrom', 'start', 'end', 'description', 'genome_version'])
RNA_TABLE['chrom'] = RNA_TABLE['chrom'].apply(format_roman_chrom)
display(RNA_TABLE)
# RNA_TABLE.to_csv('../../data/genome_annotations/sgd_database/rna_coding_R64-3-1.csv', index=False)
RNA_dict = RNA_TABLE.set_index('locus_id').to_dict()

#### INTERGENIC

intergenic_annotations = []
fasta_gz = f'{genome_annot_sgd_folder}/NotFeature_R64-3-1_20210421.fasta.gz'
with gzip.open(fasta_gz, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ID, location, genome_version, description = record.description.split(', ')
        intergenic_id = ID
        res = re.findall(r'(Chr [A-Za-z]+) from ([0-9]+)-([0-9]+)', str(location))[0]
        Chr = res[0].upper().replace(' ','_').replace('MITO', 'MT')
        Start = res[1]
        End = res[2]
        
        intergenic_annotations.append([intergenic_id, Chr, Start, End, description, genome_version])

INTERGENIC_TABLE = pd.DataFrame(intergenic_annotations, columns=['locus_id', 'chrom', 'start', 'end', 'description', 'genome_version'])
INTERGENIC_TABLE['chrom'] = INTERGENIC_TABLE['chrom'].apply(format_roman_chrom)
display(INTERGENIC_TABLE)
# INTERGENIC_TABLE.to_csv('../../data/genome_annotations/sgd_database/notFeature_R64-3-1.csv', index=False)
INTERGENIC_dict = INTERGENIC_TABLE.set_index('locus_id').to_dict()

#### CDS 
cds_annotations = []
fasta_gz = f'{genome_annot_sgd_folder}/orf_coding_all_R64-3-1_20210421.fasta.gz'
with gzip.open(fasta_gz, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ID, location, genome_version, cds_type = record.description.split(', ')[:4]
        cds_id = ID.split(' ')[0]
        cds_name = ID.split(' ')[1]
        sgd_id = ID.split(' ')[2]
        res = re.findall(r'(Chr [A-Za-z]+) from ([0-9]+)-([0-9]+)', str(location))[0]
        Chr = res[0].upper().replace(' ','_').replace('MITO', 'M')
        description = ' ; '.join(record.description.split(', ')[4:])
        if 'reverse complement' == cds_type:
            Start = res[2]
            End = res[1]
            cds_type = description.split(' ; ')[0]
            description = ' ; '.join(description.split(' ; ')[1:])
        else : 
            Start = res[1]
            End = res[2]
        
        cds_annotations.append([cds_id, cds_name, sgd_id, cds_type, Chr, Start, End, description, genome_version])
        
CDS_TABLE = pd.DataFrame(cds_annotations, columns=['locus_id', 'name', 'sgd_id', 'genome_annotations', 'chrom', 'start', 'end', 'description', 'genome_version'])
CDS_TABLE['chrom'] = CDS_TABLE['chrom'].apply(format_roman_chrom)
display(CDS_TABLE)
# CDS_TABLE.to_csv('../../data/genome_annotations/sgd_database/orf_coding_R64-3-1.csv', index=False)
CDS_dict = CDS_TABLE.set_index('locus_id').to_dict()


#### 1KB-AWAY

gene_annotations = []
fasta_gz = f'{genome_annot_sgd_folder}/orf_genomic_1000_all.fasta.gz'
with gzip.open(fasta_gz, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        ID, location, genome_version, cds_type = record.description.split(', ')[:4]
        cds_id = ID.split(' ')[0]
        cds_name = ID.split(' ')[1]
        sgd_id = ID.split(' ')[2]
        res = re.findall(r'(Chr [A-Za-z]+) from ([0-9]+)-([0-9]+)', str(location))[0]
        Chr = res[0].upper().replace(' ','_').replace('MITO', 'MT')
        description = ' ; '.join(record.description.split(', ')[4:])
        if 'reverse complement' == cds_type:
            Start = res[2]
            End = res[1]
            cds_type = description.split(' ; ')[0]
            description = ' ; '.join(description.split(' ; ')[1:])
        else : 
            Start = res[1]
            End = res[2]
        
        gene_annotations.append([cds_id, cds_name, sgd_id, cds_type, Chr, Start, End, description, genome_version])
        
GENE_TABLE = pd.DataFrame(gene_annotations, columns=['locus_id', 'name', 'sgd_id', 'genome_annotations', 'chrom', 'start', 'end', 'description', 'genome_version'])
GENE_TABLE['chrom'] = GENE_TABLE['chrom'].apply(format_roman_chrom)
display(GENE_TABLE)
# GENE_TABLE.to_csv('../../data/genome_annotations/sgd_database/orf_genomic_1000_R64-3-1.csv', index=False)
GENE_dict = GENE_TABLE.set_index('locus_id').to_dict()

,locus_id,name,sgd_id,genome_annotations,chrom,start,end,description,genome_version
0,TEL01L,TEL01L,SGDID:S000028862,reverse complement,CHR_1,1,801,"""Telomeric region on the left arm of Chromosom...",Genome Release 64-3-1
1,ARS102,ARS102,SGDID:S000121252,ARS,CHR_1,707,776,"""Autonomously Replicating Sequence""",Genome Release 64-3-1
2,ARS103,ARS103,SGDID:S000121253,ARS,CHR_1,7997,8547,"""Autonomously Replicating Sequence; replicatio...",Genome Release 64-3-1
3,YALWdelta1,YALWdelta1,SGDID:S000006787,long_terminal_repeat,CHR_1,22230,22552,"""Ty1 LTR""",Genome Release 64-3-1
4,ARS104,ARS104,SGDID:S000118317,ARS,CHR_1,30946,31183,"""Autonomously Replicating Sequence""",Genome Release 64-3-1
...,...,...,...,...,...,...,...,...,...
853,ORI2,ORI2,SGDID:S000029668,origin_of_replication,CHR_MT,32231,32501,"""Mitochondrial origin of replication""",Genome Release 64-3-1
854,ORI6,ORI6,SGDID:S000029672,origin_of_replication,CHR_MT,45227,47927,"""Mitochondrial origin of replication""",Genome Release 64-3-1
855,ORI3,ORI3,SGDID:S000029669,reverse complement,CHR_MT,54567,54840,"""Mitochondrial origin of replication""",Genome Release 64-3-1
856,ORI4,ORI4,SGDID:S000029670,reverse complement,CHR_MT,56567,56832,"""Mitochondrial origin of replication""",Genome Release 64-3-1


,locus_id,name,sgd_id,genome_annotations,chrom,start,end,description,genome_version
0,YNCA0001W,HRA1,SGDID:S000119380,ncRNA_gene,CHR_1,99305,99868,"""Non-protein-coding RNA; substrate of RNase P ...",Genome Release 64-3-1
1,YNCA0002W,TRN1,SGDID:S000006680,tRNA_gene,CHR_1,139152,139187,"""Proline tRNA (tRNA-Pro) ; predicted by tRNAsc...",Genome Release 64-3-1
2,YNCA0003W,SNR18,SGDID:S000007500,snoRNA_gene,CHR_1,142367,142468,"""C/D box small nucleolar RNA (snoRNA); commonl...",Genome Release 64-3-1
3,YNCA0004W,TGA1,SGDID:S000006521,tRNA_gene,CHR_1,166267,166339,"""Alanine tRNA (tRNA-Ala) ; predicted by tRNAsc...",Genome Release 64-3-1
4,YNCA0005W,SUP56,SGDID:S000006636,tRNA_gene,CHR_1,181141,181178,"""Leucine tRNA (tRNA-Leu) ; predicted by tRNAsc...",Genome Release 64-3-1
...,...,...,...,...,...,...,...,...,...
417,YNCQ0023W,YNCQ0023W,SGDID:S000007319,tRNA_gene,CHR_MT,77431,77505,"""Mitochondrial phenylalanine tRNA (tRNA-Phe); ...",Genome Release 64-3-1
418,YNCQ0024C,YNCQ0024C,SGDID:S000007335,tRNA_gene,CHR_MT,78089,78162,"tRNA_gene ; ""Mitochondrial threonine tRNA (tRN...",Genome Release 64-3-1
419,YNCQ0025W,YNCQ0025W,SGDID:S000007336,tRNA_gene,CHR_MT,78533,78608,"""Mitochondrial valine tRNA (tRNA-Val); predict...",Genome Release 64-3-1
420,YNCQ0026W,YNCQ0026W,SGDID:S000007326,tRNA_gene,CHR_MT,85035,85112,"""Mitochondrial formylated methionine tRNA (tRN...",Genome Release 64-3-1


,locus_id,chrom,start,end,description,genome_version
0,A:802-1806,CHR_1,802,1806,between TEL01L and YAL068C,Genome Release 64-3-1
1,A:2170-2479,CHR_1,2170,2479,between YAL068C and YAL067W-A,Genome Release 64-3-1
2,A:2708-7234,CHR_1,2708,7234,between YAL067W-A and YAL067C,Genome Release 64-3-1
3,A:9017-10090,CHR_1,9017,10090,between YAL067C and YAL066W,Genome Release 64-3-1
4,A:10400-11564,CHR_1,10400,11564,between YAL066W and YAL065C,Genome Release 64-3-1
...,...,...,...,...,...,...
6664,Q:78163-78532,CHR_MT,78163,78532,between YNCQ0024C and YNCQ0025W,Genome Release 64-3-1
6665,Q:78609-79212,CHR_MT,78609,79212,between YNCQ0025W and Q0275,Genome Release 64-3-1
6666,Q:80023-82328,CHR_MT,80023,82328,between Q0275 and ORI5,Genome Release 64-3-1
6667,Q:82601-85034,CHR_MT,82601,85034,between ORI5 and YNCQ0026W,Genome Release 64-3-1


,locus_id,name,sgd_id,genome_annotations,chrom,start,end,description,genome_version
0,YAL069W,YAL069W,SGDID:S000002143,Dubious ORF,CHR_1,335,649,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
1,YAL068W-A,YAL068W-A,SGDID:S000028594,Dubious ORF,CHR_1,538,792,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
2,YAL068C,PAU8,SGDID:S000002142,Verified ORF,CHR_1,1807,2169,"""Protein of unknown function; member of the se...",Genome Release 64-3-1
3,YAL067W-A,YAL067W-A,SGDID:S000028593,Uncharacterized ORF,CHR_1,2480,2707,"""Putative protein of unknown function; identif...",Genome Release 64-3-1
4,YAL067C,SEO1,SGDID:S000000062,Verified ORF,CHR_1,7235,9016,"""Putative permease; member of the allantoate t...",Genome Release 64-3-1
...,...,...,...,...,...,...,...,...,...
6711,Q0182,Q0182,SGDID:S000007280,Dubious ORF,CHR_1000,65770,66174,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
6712,Q0250,COX2,SGDID:S000007281,Verified ORF,CHR_1000,73758,74513,"""Subunit II of cytochrome c oxidase (Complex I...",Genome Release 64-3-1
6713,Q0255,Q0255,SGDID:S000007282,Uncharacterized ORF,CHR_1000,74495,75622,"""Maturase-like protein""",Genome Release 64-3-1
6714,Q0275,COX3,SGDID:S000007283,Verified ORF,CHR_1000,79213,80022,"""Subunit III of cytochrome c oxidase (Complex ...",Genome Release 64-3-1


,locus_id,name,sgd_id,genome_annotations,chrom,start,end,description,genome_version
0,YAL069W,YAL069W,SGDID:S000002143,Dubious ORF,CHR_1,1,1649,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
1,YAL068W-A,YAL068W-A,SGDID:S000028594,Dubious ORF,CHR_1,1,1792,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
2,YAL068C,PAU8,SGDID:S000002142,Verified ORF,CHR_1,807,3169,"""Protein of unknown function; member of the se...",Genome Release 64-3-1
3,YAL067W-A,YAL067W-A,SGDID:S000028593,Uncharacterized ORF,CHR_1,1480,3707,"""Putative protein of unknown function; identif...",Genome Release 64-3-1
4,YAL067C,SEO1,SGDID:S000000062,Verified ORF,CHR_1,6235,10016,"""Putative permease; member of the allantoate t...",Genome Release 64-3-1
...,...,...,...,...,...,...,...,...,...
6711,Q0182,Q0182,SGDID:S000007280,Dubious ORF,CHR_MT,64770,67174,"""Dubious open reading frame; unlikely to encod...",Genome Release 64-3-1
6712,Q0250,COX2,SGDID:S000007281,Verified ORF,CHR_MT,72758,75513,"""Subunit II of cytochrome c oxidase (Complex I...",Genome Release 64-3-1
6713,Q0255,Q0255,SGDID:S000007282,Uncharacterized ORF,CHR_MT,73495,76984,"""Maturase-like protein""",Genome Release 64-3-1
6714,Q0275,COX3,SGDID:S000007283,Verified ORF,CHR_MT,78213,81022,"""Subunit III of cytochrome c oxidase (Complex ...",Genome Release 64-3-1


In [5]:
OTHERS_TABLE = pd.read_csv('../../data/genome_annotations/sgd_database/other_features_genomic_R64-3-1.csv')
OTHERS_dict = OTHERS_TABLE.set_index('locus_id').to_dict()

RNA_TABLE = pd.read_csv('../../data/genome_annotations/sgd_database/rna_coding_R64-3-1.csv')
RNA_dict = RNA_TABLE.set_index('locus_id').to_dict()

INTERGENIC_TABLE = pd.read_csv('../../data/genome_annotations/sgd_database/notFeature_R64-3-1.csv')
INTERGENIC_dict = INTERGENIC_TABLE.set_index('locus_id').to_dict()

CDS_TABLE = pd.read_csv('../../data/genome_annotations/sgd_database/orf_coding_R64-3-1.csv')
CDS_dict = CDS_TABLE.set_index('locus_id').to_dict()

GENE_TABLE = pd.read_csv('../../data/genome_annotations/sgd_database/orf_genomic_1000_R64-3-1.csv')
GENE_dict = GENE_TABLE.set_index('locus_id').to_dict()

In [7]:
#### PROMOTERS 
PROMOTER_TABLE = pd.read_csv(f'{genome_annot_rossi_folder}/nfr_ndr_feature_positions.csv').rename(columns={"Chromosome_Name":"chrom", "systematic_id":"locus_id", "common_name":"name", "NFR/NDR_Start":"start", "NFR/NDR_End":"end", "nucleosome_stability":"description"})
PROMOTER_TABLE['chrom'] = PROMOTER_TABLE['chrom'].apply(format_chrom)
PROMOTER_TABLE['start'] = PROMOTER_TABLE['start'].astype(int)
PROMOTER_TABLE['end'] = PROMOTER_TABLE['end'].astype(int)
PROMOTER_dict = PROMOTER_TABLE[['locus_id', 'name', 'chrom','strand','start', 'end', 'description']].set_index('locus_id').to_dict()


#### UNSTABLE TRANSCRIPTS 2009
CUTs_2009_annotation = gffpd.read_gff3('../../data/genome_annotations/xu_2009/Xu_2009_CUTs_V64.gff3')
CUTs_2009_annotation = CUTs_2009_annotation.df.rename(columns={'seq_id':'chrom'})
CUTs_2009_annotation['chrom'] = [ CUTs_2009_annotation['chrom'][idx].replace("chr", 'CHR_') for idx in CUTs_2009_annotation.index ]
CUTs_2009_annotation['locus_id'] = [ CUTs_2009_annotation['attributes'][idx].split(';')[1].replace("Name=", '') for idx in CUTs_2009_annotation.index ]
CUTs_2009_annotation['description'] = [ 'CUT' for idx in CUTs_2009_annotation.index ]
CUTs_2009_annotation['chrom'] = CUTs_2009_annotation['chrom'].apply(format_roman_chrom)
CUT_dict = CUTs_2009_annotation[['locus_id', 'chrom','strand','start', 'end', 'description']].set_index('locus_id').to_dict()


SUTs_2009_annotation = gffpd.read_gff3('../../data/genome_annotations/xu_2009/Xu_2009_SUTs_V64.gff3')
SUTs_2009_annotation = SUTs_2009_annotation.df.rename(columns={'seq_id':'chrom'})
SUTs_2009_annotation['chrom'] = [ SUTs_2009_annotation['chrom'][idx].replace("chr", 'CHR_') for idx in SUTs_2009_annotation.index ]
SUTs_2009_annotation['locus_id'] = [ SUTs_2009_annotation['attributes'][idx].split(';')[1].replace("Name=", '') for idx in SUTs_2009_annotation.index ]
SUTs_2009_annotation['description'] = [ 'SUT' for idx in SUTs_2009_annotation.index ]
SUTs_2009_annotation['chrom'] = SUTs_2009_annotation['chrom'].apply(format_roman_chrom)
SUT_dict = SUTs_2009_annotation[['locus_id', 'chrom','strand','start', 'end', 'description']].set_index('locus_id').to_dict()

#### UNSTABLE TRANSCRIPTS 2011
XUTs_2011_annotation = gffpd.read_gff3('../../data/genome_annotations/van_dikj_2011/van_Dijk_2011_XUTs_V64.gff3')
XUTs_2011_annotation = XUTs_2011_annotation.df.rename(columns={'seq_id':'chrom'})
XUTs_2011_annotation['chrom'] = [ XUTs_2011_annotation['chrom'][idx].replace("chr", 'CHR_') for idx in XUTs_2011_annotation.index ]
XUTs_2011_annotation['locus_id'] = [ XUTs_2011_annotation['attributes'][idx].split(';')[0].replace("ID=", 'XUT_') for idx in XUTs_2011_annotation.index ]
XUTs_2011_annotation['description'] = [ XUTs_2011_annotation['attributes'][idx].split(';')[-1].replace("desc=", '') for idx in XUTs_2011_annotation.index ]
XUTs_2011_annotation['chrom'] = XUTs_2011_annotation['chrom'].apply(format_roman_chrom)
XUT_dict = XUTs_2011_annotation[['locus_id', 'chrom','strand','start', 'end', 'description']].set_index('locus_id').to_dict()

_____

## Build annotation table

In [8]:
def check_position(annotation_type, annotation_dict, candidate, snp_pos):
    if 'Promoter' in annotation_type : 
        cdd_strand = annotation_dict['strand'][candidate]
        if cdd_strand == '+':
            cdd_start = annotation_dict['start'][candidate]
            cdd_end = annotation_dict['end'][candidate]
        elif cdd_strand == '-':
            cdd_start = annotation_dict['end'][candidate]
            cdd_end = annotation_dict['start'][candidate]
    elif 'Unstable Transcript' in annotation_type : 
        cdd_strand = annotation_dict['strand'][candidate]
        if cdd_strand == '+':
            cdd_start = annotation_dict['start'][candidate]
            cdd_end = annotation_dict['end'][candidate]
        elif cdd_strand == '-':
            cdd_start = annotation_dict['end'][candidate]
            cdd_end = annotation_dict['start'][candidate]
    else :
        cdd_start = annotation_dict['start'][candidate]
        cdd_end = annotation_dict['end'][candidate]
        if '1kb-away' in annotation_type :
            distance_start = np.abs(int(snp_pos) - int(cdd_start))
            distance_end = np.abs(int(snp_pos) - int(cdd_end))
            if min([distance_start,distance_end]) == distance_start:
                annotation_type = 'Close to 5\'-UTR'
            if min([distance_start,distance_end]) == distance_end:
                annotation_type = 'Close to 3\'-UTR'

    if (int(snp_pos) >= int(cdd_start)) & (int(snp_pos) <= int(cdd_end)):
        return True, annotation_type 
    return False, annotation_type


def isAnnotated(snp_id, args):
    snps_dict = args[0]
    annotation_dict = args[1]
    snps_class = args[2]

    snp_chrom = snps_dict['chrom'][snp_id]
    snp_pos = snps_dict['position'][snp_id]
    
    candidates = []
    for key, val in annotation_dict['chrom'].items():
        if snp_chrom == val : 
            candidates.append(key)
    for cdt in candidates :
        RES, snps_class = check_position(snps_class, annotation_dict, cdt, snp_pos)
        if RES == True :
            try: 
                return (int(snp_id), cdt, annotation_dict["name"][cdt], annotation_dict["sgd_id"][cdt], annotation_dict["description"][cdt], snps_class, annotation_dict["genome_annotations"][cdt])
            except: 
                try :
                    return (int(snp_id), cdt, np.nan, np.nan,  annotation_dict["description"][cdt], snps_class, snps_class)
                except:
                    return (int(snp_id), cdt, np.nan, np.nan,  np.nan, snps_class, snps_class)
    return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan)

In [9]:
all_snps = pd.read_csv('../../data/genotype_information/yeast_snps_loc_dec2022.txt') #### need to be generated by 00a_generate_genotype_matrix.py
all_snps['position'] = all_snps['position'].replace(0,1)
all_snps['chrom'] = all_snps['chrom'].apply(format_chrom)
all_snps_dict = all_snps.set_index('snp_id').to_dict()

In [10]:
annotations = []
for snp_id in tqdm(all_snps_dict['chrom'].keys()):
    annotations.append(isAnnotated(snp_id, (all_snps_dict, INTERGENIC_dict, 'Intergenic region')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, GENE_dict, '1kb-away')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, PROMOTER_dict, 'Promoter')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, CDS_dict, 'ORF')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, RNA_dict, 'Non-coding RNA')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, CUT_dict, 'CUT')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, SUT_dict, 'SUT')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, XUT_dict, 'XUT')))
    annotations.append(isAnnotated(snp_id, (all_snps_dict, OTHERS_dict, 'Other features')))

100%|██████████| 12054/12054 [00:20<00:00, 583.25it/s]


### Multiple entries for annotation table

In [11]:
ALL_ANNOTATIONS = pd.DataFrame(annotations, columns=['snp_id', 'locus_id', 'name', 'sgd_id', 'description','snps_class_up', 'genome_annotations']).dropna(how='all').reset_index(drop=True)

In [12]:
ALL_ANNOTATIONS

,snp_id,locus_id,name,sgd_id,description,snps_class_up,genome_annotations
0,1.0,A:802-1806,NaN,NaN,between TEL01L and YAL068C,Intergenic region,Intergenic region
1,1.0,YAL069W,YAL069W,SGDID:S000002143,"""Dubious open reading frame; unlikely to encod...",Close to 3'-UTR,Dubious ORF
2,1.0,X1L_NFR/NDR,NaN,NaN,NaN,Promoter,Promoter
3,2.0,A:802-1806,NaN,NaN,between TEL01L and YAL068C,Intergenic region,Intergenic region
4,2.0,YAL069W,YAL069W,SGDID:S000002143,"""Dubious open reading frame; unlikely to encod...",Close to 3'-UTR,Dubious ORF
...,...,...,...,...,...,...,...
29929,12050.0,Q0250,COX2,SGDID:S000007281,"""Subunit II of cytochrome c oxidase (Complex I...",Close to 3'-UTR,Verified ORF
29930,12051.0,Q0250,COX2,SGDID:S000007281,"""Subunit II of cytochrome c oxidase (Complex I...",Close to 3'-UTR,Verified ORF
29931,12052.0,Q:77506-78088,NaN,NaN,between YNCQ0023W and YNCQ0024C,Intergenic region,Intergenic region
29932,12053.0,Q:82601-85034,NaN,NaN,between ORI5 and YNCQ0026W,Intergenic region,Intergenic region


In [15]:
snp_annotations = [] 
for snp_id in np.unique(ALL_ANNOTATIONS['snp_id']):
    TMP = ALL_ANNOTATIONS[ALL_ANNOTATIONS['snp_id'] == snp_id]
    snps_annots = []
    for snps_class in TMP['snps_class_up'] :
        if ('ORF' in snps_class) or ('CUT' in snps_class) or ('SUT' in snps_class) or ('XUT' in snps_class):
            snps_annots.append(snps_class)
    snps_annots = '_'.join(snps_annots)
    snp_annotations.append([int(snp_id), snps_annots])

In [16]:
annotation_stats = pd.DataFrame(snp_annotations, columns=['snp_id', 'snps_annotations']) 

### Save Unstable Transcript Annotations

In [17]:
np.unique(annotation_stats['snps_annotations'])

array(['', 'CUT', 'CUT_SUT', 'CUT_SUT_XUT', 'CUT_XUT', 'ORF', 'ORF_CUT',
       'ORF_CUT_XUT', 'ORF_SUT', 'ORF_SUT_XUT', 'ORF_XUT', 'SUT',
       'SUT_XUT', 'XUT'], dtype=object)

In [19]:
annotation_stats[annotation_stats['snps_annotations'] != ''].to_csv('../../data/genotype_information/snps_annotations_genome-version-3-64-1_UTs.txt', index=False)

In [20]:
def get_gene_name(x):
    try : 
        return GENE_TABLE[GENE_TABLE['locus_id'] == x ]['name'].values[0]
    except : 
        return ALL_ANNOTATIONS[ALL_ANNOTATIONS['locus_id'] == x ]['name'].values[0]

## Unique entry for annotation table

In [21]:
ALL_ANNOTATIONS = ALL_ANNOTATIONS.drop_duplicates(subset=['snp_id'], keep='last')

In [22]:
ALL_ANNOTATIONS['name'] = ALL_ANNOTATIONS['locus_id'].apply(get_gene_name)

/tmp/ipykernel_673711/3740738677.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ALL_ANNOTATIONS['name'] = ALL_ANNOTATIONS['locus_id'].apply(get_gene_name)


In [23]:
ALL_ANNOTATIONS

,snp_id,locus_id,name,sgd_id,description,snps_class_up,genome_annotations
2,1.0,X1L_NFR/NDR,NaN,NaN,NaN,Promoter,Promoter
5,2.0,X1L_NFR/NDR,NaN,NaN,NaN,Promoter,Promoter
7,3.0,YAL069W,YAL069W,SGDID:S000002143,"""Dubious open reading frame; unlikely to encod...",Close to 3'-UTR,Dubious ORF
9,4.0,YAL069W,YAL069W,SGDID:S000002143,"""Dubious open reading frame; unlikely to encod...",Close to 3'-UTR,Dubious ORF
11,5.0,YAL069W,YAL069W,SGDID:S000002143,"""Dubious open reading frame; unlikely to encod...",Close to 3'-UTR,Dubious ORF
...,...,...,...,...,...,...,...
29929,12050.0,Q0250,COX2,SGDID:S000007281,"""Subunit II of cytochrome c oxidase (Complex I...",Close to 3'-UTR,Verified ORF
29930,12051.0,Q0250,COX2,SGDID:S000007281,"""Subunit II of cytochrome c oxidase (Complex I...",Close to 3'-UTR,Verified ORF
29931,12052.0,Q:77506-78088,NaN,NaN,between YNCQ0023W and YNCQ0024C,Intergenic region,Intergenic region
29932,12053.0,Q:82601-85034,NaN,NaN,between ORI5 and YNCQ0026W,Intergenic region,Intergenic region


In [24]:
ALL_ANNOTATIONS.groupby('snps_class_up').count()

,snp_id,locus_id,name,sgd_id,description,genome_annotations
snps_class_up,,,,,,
CUT,287,287,0,0,287,287
Close to 3'-UTR,1598,1598,1598,1598,1598,1598
Intergenic region,92,92,0,0,92,92
Non-coding RNA,42,42,42,42,42,42
ORF,5552,5552,5552,5552,5552,5552
Other features,477,477,477,477,477,477
Promoter,1901,1901,0,0,1422,1901
SUT,428,428,0,0,428,428
XUT,1677,1677,0,0,1677,1677


In [25]:
def get_snps_class_down(x):
    locus_id = ALL_ANNOTATIONS[ALL_ANNOTATIONS['snp_id'] == x]['locus_id'].values[0]
    snps_class = ALL_ANNOTATIONS[ALL_ANNOTATIONS['snp_id'] == x]['snps_class_up'].values[0]
    genome_annotations = ALL_ANNOTATIONS[ALL_ANNOTATIONS['snp_id'] == x]['genome_annotations'].values[0]
    if snps_class == 'ORF' :
        if '_' in genome_annotations :
            genome_annotations = genome_annotations.replace('_',' ')
        return genome_annotations
    if snps_class == 'Intergenic region':
        return 'Intergenic region'
    if snps_class == 'Close to 5\'-UTR':
        return 'Close to 5\'-UTR'
    if snps_class == 'Close to 3\'-UTR':
        return 'Close to 3\'-UTR'
    if snps_class == 'Promoter':
        return 'Promoter'
    if snps_class == 'Non-coding RNA':
        if '_' in genome_annotations :
            genome_annotations = genome_annotations.replace('_',' ')
        return genome_annotations
    if 'UT' in locus_id : 
        try : 
            x = re.findall(r'^([CSXUT]{3})', locus_id)[0]
            return x 
        except: 
            return locus_id 
    if 'ARS' in locus_id : 
        return 'Replication' 
    if 'CEN' in locus_id : 
        return "Centromere"
    if 'TEL' in locus_id : 
        return "Telomere"
    if 'NTS1-2' in locus_id :
        return 'Mating-related region'
    if 'RE' in locus_id :
        return 'Recombination enhancer'
    if 'HM' in locus_id :
        return "Mating-related region"
    if 'MATALPHA' in locus_id :
        return "Mating-related region"
    else : 
        try : 
            res = re.findall(r'(^[A-Z]+[a-z]+)', locus_id)
            if len(res) == 1 :
                if genome_annotations == "LTR_retrotransposon":
                    return "LTR retrotransposon"
                else : 
                    return "LTR"
        except :
            return x 
    return np.nan

In [26]:
ALL_ANNOTATIONS["snps_class_down"] = ALL_ANNOTATIONS['snp_id'].progress_apply(get_snps_class_down)

100%|██████████| 12054/12054 [00:07<00:00, 1564.39it/s]
/tmp/ipykernel_673711/650163950.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ALL_ANNOTATIONS["snps_class_down"] = ALL_ANNOTATIONS['snp_id'].progress_apply(get_snps_class_down)


In [27]:
ALL_ANNOTATIONS.groupby("snps_class_up").count()

,snp_id,locus_id,name,sgd_id,description,genome_annotations,snps_class_down
snps_class_up,,,,,,,
CUT,287,287,0,0,287,287,287
Close to 3'-UTR,1598,1598,1598,1598,1598,1598,1598
Intergenic region,92,92,0,0,92,92,92
Non-coding RNA,42,42,42,42,42,42,42
ORF,5552,5552,5552,5552,5552,5552,5552
Other features,477,477,477,477,477,477,477
Promoter,1901,1901,0,0,1422,1901,1901
SUT,428,428,0,0,428,428,428
XUT,1677,1677,0,0,1677,1677,1677


In [28]:
ALL_ANNOTATIONS.groupby("snps_class_down").count()

,snp_id,locus_id,name,sgd_id,description,snps_class_up,genome_annotations
snps_class_down,,,,,,,
CUT,287,287,0,0,287,287,287
Centromere,17,17,17,17,17,17,17
Close to 3'-UTR,1598,1598,1598,1598,1598,1598,1598
Dubious ORF,192,192,192,192,192,192,192
Intergenic region,92,92,0,0,92,92,92
LTR,161,161,161,161,161,161,161
LTR retrotransposon,14,14,14,14,14,14,14
Mating-related region,12,12,12,12,12,12,12
Promoter,1901,1901,0,0,1422,1901,1901


In [29]:
all_snps.merge(ALL_ANNOTATIONS, on='snp_id', how='outer').to_csv('../../data/genotype_information/snps_annotations_genome-version-3-64-1.txt', index=False)